# EXP-033 | Super-Ensemble — 기존 실험 제출 예측 Rank Average

새 모델 학습 없이 기존 실험의 test 예측값을 Rank Average로 앙상블.
각 실험이 서로 다른 피처셋/부스터/파라미터를 사용하여 오차 상쇄 효과 기대.

| 포함 실험 | 제출 점수 | OOF AUC | 특징 |
|---|---|---|---|
| EXP-020 | 0.74217 | 0.74068 | 기준선, FE-v1, GBDT |
| EXP-028 | 0.74215 | 0.74082 | FE-v2+TE+ITE, Optuna LGB |
| EXP-031 | (미제출) | 0.74062 | LGB DART booster |
| EXP-032 | 0.74219 | 0.74071 | 모델별 다른 피처셋 |

| 기준선 | EXP-032 제출 0.74219 (현재 최고) |

In [7]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import date
from itertools import combinations
from scipy.stats import rankdata

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/raw')
SUB_DIR  = Path('../data/submissions')
DOCS_DIR = Path('../docs')
EXP_NO   = 33
AUTHOR   = '조여진'

sub_template = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'sample_submission: {sub_template.shape}')

sample_submission: (90067, 2)


## 1. 기존 실험 test 예측 로드

In [8]:
SOURCE_FILES = {
    'EXP-020': SUB_DIR / 'submission_exp020_조여진_07407_normalized.csv',
    'EXP-028': SUB_DIR / 'submission_exp028_조여진_07408.csv',
    'EXP-031': SUB_DIR / 'submission_exp031_조여진_07406.csv',
    'EXP-032': SUB_DIR / 'submission_exp032_조여진_07407.csv',
}

preds = {}
for name, path in SOURCE_FILES.items():
    df = pd.read_csv(path)
    preds[name] = df['probability'].values
    print(f'{name}  min={preds[name].min():.4f}  max={preds[name].max():.4f}  shape={df.shape}')

exp_names = list(preds.keys())
pred_matrix = np.stack([preds[k] for k in exp_names], axis=1)
print(f'\n예측 행렬: {pred_matrix.shape}  ({pred_matrix.shape[1]}개 실험 × {pred_matrix.shape[0]}개 샘플)')

EXP-020  min=0.0000  max=1.0000  shape=(90067, 2)
EXP-028  min=0.0007  max=0.8891  shape=(90067, 2)
EXP-031  min=0.0007  max=0.8897  shape=(90067, 2)
EXP-032  min=0.0000  max=1.0000  shape=(90067, 2)

예측 행렬: (90067, 4)  (4개 실험 × 90067개 샘플)


## 2. 실험 간 예측 상관관계 확인

In [9]:
corr_df = pd.DataFrame(pred_matrix, columns=exp_names).corr()
print('예측값 상관관계 (낮을수록 앙상블 효과 큼):')
print(corr_df.round(4).to_string())

예측값 상관관계 (낮을수록 앙상블 효과 큼):
         EXP-020  EXP-028  EXP-031  EXP-032
EXP-020   1.0000   0.9611   0.9606   0.9996
EXP-028   0.9611   1.0000   0.9999   0.9609
EXP-031   0.9606   0.9999   1.0000   0.9604
EXP-032   0.9996   0.9609   0.9604   1.0000


## 3. 모든 조합 Rank Average 비교

OOF 예측값이 없으므로 test 앙상블 결과만 생성. 조합별로 저장 후 제출 점수로 판단.

In [10]:
def rank_avg(pred_cols):
    """여러 예측값의 Rank Average → 0~1 정규화"""
    ranks = np.stack([rankdata(p) for p in pred_cols], axis=1)
    avg   = ranks.mean(axis=1)
    return (avg - avg.min()) / (avg.max() - avg.min())

results = {}

# 2개 조합
for r in range(2, len(exp_names) + 1):
    for combo in combinations(range(len(exp_names)), r):
        names  = [exp_names[i] for i in combo]
        label  = '+'.join(names)
        result = rank_avg([pred_matrix[:, i] for i in combo])
        results[label] = result

BEST_SUB = 0.74219  # EXP-032 현재 최고 제출 점수

print(f'총 {len(results)}가지 조합 생성 완료')
print(f'기준선(EXP-032 제출): {BEST_SUB}')
print()
print(f'{"조합":<55} {"확률 min":>8} {"확률 max":>8}')
print('-' * 75)
for label, pred in results.items():
    print(f'  {label:<53} {pred.min():.4f}   {pred.max():.4f}')

총 11가지 조합 생성 완료
기준선(EXP-032 제출): 0.74219

조합                                                        확률 min   확률 max
---------------------------------------------------------------------------
  EXP-020+EXP-028                                       0.0000   1.0000
  EXP-020+EXP-031                                       0.0000   1.0000
  EXP-020+EXP-032                                       0.0000   1.0000
  EXP-028+EXP-031                                       0.0000   1.0000
  EXP-028+EXP-032                                       0.0000   1.0000
  EXP-031+EXP-032                                       0.0000   1.0000
  EXP-020+EXP-028+EXP-031                               0.0000   1.0000
  EXP-020+EXP-028+EXP-032                               0.0000   1.0000
  EXP-020+EXP-031+EXP-032                               0.0000   1.0000
  EXP-028+EXP-031+EXP-032                               0.0000   1.0000
  EXP-020+EXP-028+EXP-031+EXP-032                       0.0000   1.0000


## 4. 전체 4개 조합 Submission 저장 (메인)

전체 조합 앙상블이 가장 분산을 줄이는 효과가 크므로 이를 주 제출 파일로 저장.
나머지 조합도 비교를 위해 저장.

In [11]:
SUB_DIR.mkdir(parents=True, exist_ok=True)

# 전체 4개 앙상블 — 메인 제출
main_label = '+'.join(exp_names)
main_pred  = results[main_label]

sub_main = sub_template.copy()
sub_main['probability'] = main_pred
main_fname = f'submission_exp{EXP_NO:03d}_{AUTHOR}_all4.csv'
sub_main.to_csv(SUB_DIR / main_fname, index=False)
print(f'[메인] 저장: {main_fname}')
print(f'       probability 범위: [{main_pred.min():.4f}, {main_pred.max():.4f}]')
print()

# 3개 조합 — 비교용
three_combos = [label for label in results if label.count('+') == 2]
for label in three_combos:
    pred = results[label]
    short = label.replace('EXP-', '').replace('+', '_')
    fname = f'submission_exp{EXP_NO:03d}_{AUTHOR}_{short}.csv'
    s = sub_template.copy()
    s['probability'] = pred
    s.to_csv(SUB_DIR / fname, index=False)
    print(f'[비교] 저장: {fname}')

print(f'\n총 {1 + len(three_combos)}개 파일 저장 완료')
print(f'\n→ 우선 제출 추천: {main_fname}  (4개 전체 앙상블)')

[메인] 저장: submission_exp033_조여진_all4.csv
       probability 범위: [0.0000, 1.0000]

[비교] 저장: submission_exp033_조여진_020_028_031.csv
[비교] 저장: submission_exp033_조여진_020_028_032.csv
[비교] 저장: submission_exp033_조여진_020_031_032.csv
[비교] 저장: submission_exp033_조여진_028_031_032.csv

총 5개 파일 저장 완료

→ 우선 제출 추천: submission_exp033_조여진_all4.csv  (4개 전체 앙상블)


## 5. 실험 기록

In [12]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

def log_to_leaderboard(exp_no, author, model_name, params_str,
                        oof_auc, cv_strategy, preprocessing_ver, n_features,
                        imbalance_method, submitted, hackathon_score,
                        file_name, notes='', insights=''):
    lb_path = DOCS_DIR / 'leaderboard.xlsx'
    wb = load_workbook(lb_path)
    ws = wb['리더보드']
    exp_label = f'EXP-{exp_no:03d}'
    next_row = ws.max_row + 1
    for r in range(2, ws.max_row + 1):
        val = ws.cell(row=r, column=2).value
        if val == exp_label:
            next_row = r; break
        if ws.cell(row=r, column=1).value is None or str(ws.cell(row=r, column=1).value).strip() == '':
            next_row = r; break
    values = [str(date.today()), exp_label, author, model_name, params_str,
              None, None, None, None, round(oof_auc, 5) if oof_auc else None,
              cv_strategy, preprocessing_ver, n_features, imbalance_method,
              submitted, hackathon_score, file_name, notes, insights]
    thin   = Side(style='thin', color='B0B8D0')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill   = PatternFill('solid', fgColor='EEF2FA') if next_row % 2 == 0 else None
    font   = Font(name='맑은 고딕', size=10)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    left_cols = {4, 5, 12, 14, 17, 18, 19}
    for c_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=next_row, column=c_idx, value=val)
        cell.font = font; cell.border = border
        cell.alignment = left if c_idx in left_cols else center
        if fill: cell.fill = fill
        if c_idx in range(6, 11) or c_idx == 16: cell.number_format = '0.00000'
    wb.save(lb_path)
    print(f'[leaderboard.xlsx] EXP-{exp_no:03d} 기록 완료 (row {next_row})')

# test-only 앙상블이라 진짜 OOF AUC 없음 → 소스 실험 OOF AUC 평균을 대리 지표로 기록
SOURCE_OOF = {'EXP-020': 0.74068, 'EXP-028': 0.74082, 'EXP-031': 0.74062, 'EXP-032': 0.74071}
proxy_oof  = np.mean(list(SOURCE_OOF.values()))
print(f'소스 실험 OOF AUC 평균 (proxy): {proxy_oof:.5f}')
print(f'  {SOURCE_OOF}')

params_str = 'RankAvg(EXP-020+EXP-028+EXP-031+EXP-032) | 새 학습 없음, 기존 test 예측 앙상블'
NOTES    = ('Super-ensemble: 기존 4개 실험 test 예측 Rank Average. '
            'OOF AUC는 소스 실험 평균값(proxy). '
            '주의: EXP-020≈EXP-032(corr=0.9996), EXP-028≈EXP-031(corr=0.9999) — 실질적 2그룹 앙상블.')
INSIGHTS = ('EXP-020(0.74217)+EXP-028(0.74215)+EXP-031(미제출)+EXP-032(0.74219) rank avg | '
            '가장 다양한 조합: EXP-028+EXP-032(corr=0.9609) | 3개 조합 비교용 파일도 저장됨')

log_to_leaderboard(
    EXP_NO, AUTHOR, 'Super-Ensemble(RankAvg)', params_str,
    proxy_oof, 'N/A (test-only)', 'mixed', None,
    'mixed', 'N', None,
    f'notebooks/29_super_ensemble_yjcho.ipynb', NOTES, INSIGHTS
)

소스 실험 OOF AUC 평균 (proxy): 0.74071
  {'EXP-020': 0.74068, 'EXP-028': 0.74082, 'EXP-031': 0.74062, 'EXP-032': 0.74071}
[leaderboard.xlsx] EXP-033 기록 완료 (row 30)
